In [ ]:
import openai
import requests
import json
import time

# OpenAI API 키 설정
openai.api_key = 'your-api-key'

# OpenStreetMap API를 통해 장소 주소 가져오기 (재시도 포함)
def get_location_address_with_retries(place_name, max_retries=10, retry_delay=2):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": place_name,
        "format": "json",
        "addressdetails": 1,
        "limit": 1
    }
    headers = {
        "User-Agent": "MyApplication/1.0 (your_email@example.com)"  # 필수 User-Agent 헤더
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, headers=headers)

            # HTTP 상태코드 확인
            if response.status_code == 403:
                print(f"[Attempt {attempt}/{max_retries}] HTTP 403 Error (Rate Limit Exceeded). Retrying...")
                time.sleep(retry_delay)
                continue
            elif response.status_code != 200:
                print(f"[Attempt {attempt}/{max_retries}] HTTP {response.status_code} Error. Retrying...")
                time.sleep(retry_delay)
                continue

            # 응답이 비어 있는 경우 처리
            if not response.text.strip():
                print(f"[Attempt {attempt}/{max_retries}] Empty response from server. Retrying...")
                time.sleep(retry_delay)
                continue

            # JSON 파싱 시도
            data = response.json()
            if data:
                return {"name": place_name, "address": data[0]["display_name"]}

            print(f"[Attempt {attempt}/{max_retries}] Address not found. Retrying...")
            time.sleep(retry_delay)

        except requests.exceptions.RequestException as e:
            print(f"[Attempt {attempt}/{max_retries}] Network error: {e}. Retrying...")
            time.sleep(retry_delay)

        except json.JSONDecodeError:
            print(f"[Attempt {attempt}/{max_retries}] JSON decode error. Retrying...")
            time.sleep(retry_delay)

    return {"name": place_name, "address": f"Error: Unable to retrieve address after {max_retries} attempts, 더 구체적인 지역명을 말해주세요"}

# 사용자 입력으로부터 특징 추출
def extract_route_features():
    def request_additional_input(prompt):
        return input(prompt)
    
    user_input = input("Describe your running preferences: ")
    
    prompt = f"""
    You are a highly skilled and motivational running coach, dedicated to encouraging users during their running journey. Always respond with kindness, positivity, and expert advice tailored to the user's needs. Use encouraging phrases like "You're doing great!" or "Just a little more, you can do it!" to keep the user motivated.
    When providing feedback, offer actionable tips in a friendly tone, focusing on improving running technique, pace control, or endurance. Be empathetic and supportive, ensuring users feel understood and empowered. For example, if a user asks for the remaining distance, include motivational phrases like "Almost there! You're stronger than you think!"
    Your ultimate goal is to act as a trusted running mate, cheering users on and making their experience enjoyable, productive, and safe. Stay engaging and friendly, combining professionalism with warm encouragement to inspire confidence and persistence.

    The user has provided the following running-related requirements: "{user_input}".
    If the user's input is Korean, analyze the results of the English translation and extract the requirements.
    Analyze the user's input, even if it includes indirect expressions, and infer the preferences or dislikes based on the context.
    All elements received should be located only in Seoul, South Korea
    Do not add toilets as a preferred facility unless the user explicitly states that they prefer them.

    When inferring facilities:
    - Include only facilities explicitly mentioned by the user or clearly implied through their descriptions.
    - Do not infer facilities based solely on general assumptions or the tag list.
    - If there is no explicit or implicit reference to facilities, assign `facilities` and `dislike_facilities` as `null`.

    Use the following guidelines:
    1. Explicit mentions take precedence.
    2. Do not generalize based on default tags unless directly relevant.
    3. Avoid including unnecessary facilities unless specified or implied.
    4. All incoming addresses should be located only in Seoul, South Korea

    For example:
    - If the user mentions "clean air," infer environments like ["parks", "forests", "open areas"].
    - If the user mentions "dusty areas are not preferred," infer disliked facilities like ["construction sites", "industrial zones"].
    - If the user mentions "I don't like crowds," infer disliked environments like ["busy streets", "urban areas"].
    - If the user mentions "quiet and peaceful places," infer preferred environments like ["parks", "forests", "riverside"].

    Please extract the following details and preferences, adhering to the format below. If any key is not explicitly mentioned and cannot be inferred, assign it a value of `null`. Assign default values only when explicitly instructed to do so.

    ### Required JSON structure:
    {{
        "Distance": "",#Default to 3 if not mentioned
        "start": [{{"name": "", "address": ""}}],
        "destination": [{{"name": "", "address": ""}}],
        "incline": "", #Default to 'low' if not mentioned
        "environment": [""],
        "dislike_environment": [""],
        "facilities": ["", ""],
        "dislike_facilities": [""],
        "environment_distance_threshold": 300,
        "facility_distance_threshold": 300
    }}

    ### Extract the following details:
    1. Distance (in kilometers. Extract only numbers. Default to 3 if not mentioned)
    2. Start location # Mandatory. Ask the user if not mentioned.
    3. Destination location (Default to same as Start location if not mentioned)
    4. Incline preference (low, medium, high. Default to `low` if not mentioned)
    5. Preferred environments (e.g., ["water", "forest", "park"])
    6. Disliked environments (e.g., ["busy streets", "construction sites"])
    7. Preferred facilities (e.g., ["toilets", "convenience stores"])
    8. Disliked facilities (e.g., ["parking lots"])
    9. Environment distance threshold (Default to 300)
    10. Facility distance threshold (Default to 300)

    ### Important:
    - All incoming addresses should be located only in Seoul, South Korea.
    - When the user provides indirect descriptions (e.g., "clean air"), infer relevant environments (e.g., "parks" or "forests") rather than repeating the phrase.
    - Ensure all inferred values are **realistic and contextually accurate** based on the input.
    - If the user mentions experiencing physical discomfort (e.g., back pain), set `incline` to "low".
    - Return the results in **JSON format** with `null` values for any unspecified or un-inferable keys.

    Analyze the user's input thoroughly, infer all plausible details, and provide the JSON output.
    Use the following tag list when inferring environments or facilities.:
    "amenity:drinking_water", "amenity:bench", "amenity:toilets", "amenity:locker", "amenity:shower", "amenity:changing_rooms", "amenity:vending_machine", "amenity:parking", "amenity:bicycle_parking", "amenity:track", "amenity:sports_centre", "amenity:stadium", "amenity:fitness_station", "amenity:gym", "amenity:outdoor_gym", "amenity:cafeteria", "amenity:restaurant", "amenity:picnic_table", "amenity:hospital", "amenity:clinic", "amenity:viewpoint", "amenity:community_centre", "amenity:clubhouse", "amenity:event_space", "amenity:bar", "amenity:pub", "amenity:nightclub", "amenity:marketplace", "amenity:bus_station", "amenity:taxi", "amenity:fuel", "amenity:waste_disposal", "amenity:waste_transfer_station", "amenity:sewage_plant", "amenity:recycling", "amenity:industrial", "amenity:prison", "amenity:construction", "building:gym", "building:sports_centre", "building:stadium", "building:clubhouse", "building:fitness_station", "building:school", "building:university", "building:mall", "building:hotel", "building:office", "building:marketplace", "building:industrial", "building:warehouse", "building:prison", "building:construction", "geological:outcrop", "geological:rock", "geological:sedimentary", "geological:karst", "geological:exposure", "geological:cliff", "geological:landslide", "geological:fault", "geological:sinkhole", "geological:escarpment", "highway:footway", "highway:path", "highway:pedestrian", "highway:pedestrian_zone", "highway:corridor", "highway:alley", "leisure:track", "leisure:outdoor_gym", "leisure:sports_centre", "leisure:stadium", "leisure:park", "leisure:garden", "leisure:playground", "leisure:dog_park", "leisure:picnic_site", "leisure:pitch", "leisure:fishing", "natural:forest", "natural:river", "natural:stream", "natural:lake", "natural:grassland", "natural:valley", "natural:beach", "natural:ridge", "natural:meadow", "natural:hill", "natural:rock", "natural:cliff", "natural:sinkhole", "natural:scree", "natural:moraine", "natural:glacier", "natural:bare_rock", "natural:wetland", "natural:swamp", "natural:marsh", "natural:desert", "natural:sand_dune", "natural:saltmarsh", "natural:sand", "public_transport:platform", "public_transport:station", "shop:convenience", "shop:beverages", "shop:water", "shop:supermarket", "shop:pharmacy", "shop:health_food", "shop:mall", "shop:department_store", "shop:marketplace", "shop:alcohol", "shop:lottery", "shop:waste", "shop:cafe", "shop:bakery", "tourism:viewpoint", "tourism:information", "tourism:attraction", "tourism:park", "tourism:picnic_site", "tourism:hotel", "tourism:hostel", "tourism:museum", "tourism:monument", "water:river", "water:stream", "water:lake", "water:pond", "water:canal", "water:bay", "water:waterfall", "water:reservoir", "water:marsh", "water:swamp", "water:salt_marsh", "water:floodplain", "water:beach", "waterway:river", "waterway:stream", "waterway:canal", "waterway:drift", "waterway:spring", "waterway:ditch", "waterway:sluice", "waterway:weir", "waterway:drain", "waterway:wastewater"
    """

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are an assistant designed to help users by extracting specific features from their running-related requirements. Your task is to identify key attributes such as distance, inclination, environment, and any additional preferences mentioned by the user. Respond only with these details in a structured JSON format without any extra information."},
                {"role": "user", "content": prompt}
            ],
        )

        # Assuming the response message content is properly formatted JSON
        result = response['choices'][0]['message']['content'].strip()
        parsed_result = json.loads(result)

        # 필수 장소의 주소를 동적으로 가져오기
        if parsed_result.get("start"):
            for location in parsed_result["start"]:
                address_info = get_location_address_with_retries(location["name"])
                location.update(address_info) # 주소 업데이트

        # destination이 없으면 start와 동일하게 설정
        if not parsed_result.get("destination") or not parsed_result["destination"][0].get("name"):
            parsed_result["destination"] = parsed_result["start"]
        else:
            for location in parsed_result["destination"]:
                address_info = get_location_address_with_retries(location["name"])
                location.update(address_info)
                
        # 사용자 입력 추가 수집 로직
        while True:
            count_non_null = sum(
                1 for key in ["incline", "environment", "dislike_environment", "facilities", "dislike_facilities"]
                if parsed_result[key] != "null" and parsed_result[key]
            )
            
            if count_non_null >= 2:
                break
            
            # 부족한 정보를 사용자에게 요청
            if not parsed_result.get("incline") or parsed_result["incline"] == "null":
                parsed_result["incline"] = request_additional_input("What is your incline preference (low, medium, high)? If you have no preference, type 'null': ")

            if not parsed_result.get("environment") or not parsed_result["environment"]:
                env = request_additional_input("What kind of environments do you prefer for running? (e.g., park, forest): ")
                parsed_result["environment"] = [e.strip() for e in env.split(',') if e.strip()]

            if not parsed_result.get("dislike_environment") or not parsed_result["dislike_environment"]:
                dislike_env = request_additional_input("Are there any environments you dislike for running? (e.g., busy streets, industrial areas): ")
                parsed_result["dislike_environment"] = [e.strip() for e in dislike_env.split(',') if e.strip()]

            if not parsed_result.get("facilities") or not parsed_result["facilities"]:
                facilities = request_additional_input("What facilities do you prefer along your running route? (e.g., water fountains, benches): ")
                parsed_result["facilities"] = [f.strip() for f in facilities.split(',') if f.strip()]

            if not parsed_result.get("dislike_facilities") or not parsed_result["dislike_facilities"]:
                dislike_facilities = request_additional_input("Are there any facilities you dislike along your running route? (e.g., parking lots, construction sites): ")
                parsed_result["dislike_facilities"] = [f.strip() for f in dislike_facilities.split(',') if f.strip()]

        return parsed_result
    except Exception as e:
        return {"error": str(e)}

In [21]:
# Example usage
#user_input = "I want to run around 3km in the direction of a park. I would like toilets along the route, and a convenience store where I can get water would be nice. I prefer a low incline. I dislike areas with parking lots. I want to start from sangam middel school"
#user_input = "I want to run 3km on a flat road with convenience and park and toilet starting 마포중앙도서관."
#user_input ="I want to run around 3km in the direction of a park. I would like toilets along the route, and a convenience store where I can get water would be nice. I prefer a low incline. I dislike areas with parking lots. I want to start from sangam elementary and end to 이화여자대학교."
# user_input ="상암초등학교에서 출발해 이화여대까지 뛰고싶어. 경사는 심하진 않았으면 좋겠고 공기가 맑은 곳에서 뛰고싶어"
#user_input ="조용한 곳에서 달리고 싶어요. 출발지는 이화여대, 도착지는 여의도한강공원으로 하고 싶어요."
#user_input ="3km 정도 달릴 예정인데, 공기가 깨끗한 공원이나 숲 근처가 좋겠어요. 복잡한 도심은 피하고 싶습니다."
#user_input ="5km를 뛰고 싶어요. 출발은 홍대, 도착은 이화여대. 중간에 화장실과 음수대가 있는 경로로 안내해주세요."#화장실이나 음수대면 하나만 뽑기도 함
#user_input ="상암동에서 출발해서 4km 이내로 달리고 싶어요. 경사가 낮은 코스를 추천해주세요. 공사장 근처는 피하고 싶어요."
#user_input ="이화여자대학교에서 출발해서 2km 정도 달릴 거예요. 숲 근처로 조용한 경로가 좋겠어요. 편의점이나 주차장 근처는 선호하지 않아요."
#user_input = "3km 달리고 싶어요. 한강 근처, 출발지는 이대역으로 해주세요."

features = extract_route_features()

In [22]:
print(json.dumps(features, indent=2, ensure_ascii=False))

{
  "Distance": "2",
  "start": [
    {
      "name": "Ewha Womans University",
      "address": "이화여자대학교, 52, 이화여대길, 대현동, 신촌동, 서대문구, 서울특별시, 03760, 대한민국"
    }
  ],
  "destination": [
    {
      "name": "Ewha Womans University",
      "address": "이화여자대학교, 52, 이화여대길, 대현동, 신촌동, 서대문구, 서울특별시, 03760, 대한민국"
    }
  ],
  "incline": "low",
  "environment": [
    "forests"
  ],
  "dislike_environment": [],
  "facilities": [],
  "dislike_facilities": [],
  "environment_distance_threshold": 300,
  "facility_distance_threshold": 300
}


In [23]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# NLP 모델 로드
model = SentenceTransformer('all-mpnet-base-v2')

# 태그 리스트
osm_tags = [
"amenity:drinking_water", "amenity:bench", "amenity:toilets", "amenity:locker", "amenity:shower", "amenity:changing_rooms", "amenity:vending_machine", "amenity:parking", "amenity:bicycle_parking", "amenity:track", "amenity:sports_centre", "amenity:stadium", "amenity:fitness_station", "amenity:gym", "amenity:outdoor_gym", "amenity:cafeteria", "amenity:restaurant", "amenity:picnic_table", "amenity:hospital", "amenity:clinic", "amenity:viewpoint", "amenity:community_centre", "amenity:clubhouse", "amenity:event_space", "amenity:bar", "amenity:pub", "amenity:nightclub", "amenity:marketplace", "amenity:bus_station", "amenity:taxi", "amenity:fuel", "amenity:waste_disposal", "amenity:waste_transfer_station", "amenity:sewage_plant", "amenity:recycling", "amenity:industrial", "amenity:prison", "amenity:construction", "building:gym", "building:sports_centre", "building:stadium", "building:clubhouse", "building:fitness_station", "building:school", "building:university", "building:mall", "building:hotel", "building:office", "building:marketplace", "building:industrial", "building:warehouse", "building:prison", "building:construction", "geological:outcrop", "geological:rock", "geological:sedimentary", "geological:karst", "geological:exposure", "geological:cliff", "geological:landslide", "geological:fault", "geological:sinkhole", "geological:escarpment", "highway:footway", "highway:path", "highway:pedestrian", "highway:pedestrian_zone", "highway:corridor", "highway:alley", "leisure:track", "leisure:outdoor_gym", "leisure:sports_centre", "leisure:stadium", "leisure:park", "leisure:garden", "leisure:playground", "leisure:dog_park", "leisure:picnic_site", "leisure:pitch", "leisure:fishing", "natural:forest", "natural:river", "natural:stream", "natural:lake", "natural:grassland", "natural:valley", "natural:beach", "natural:ridge", "natural:meadow", "natural:hill", "natural:rock", "natural:cliff", "natural:sinkhole", "natural:scree", "natural:moraine", "natural:glacier", "natural:bare_rock", "natural:wetland", "natural:swamp", "natural:marsh", "natural:desert", "natural:sand_dune", "natural:saltmarsh", "natural:sand", "public_transport:platform", "public_transport:station", "shop:convenience", "shop:beverages", "shop:water", "shop:supermarket", "shop:pharmacy", "shop:health_food", "shop:mall", "shop:department_store", "shop:marketplace", "shop:alcohol", "shop:lottery", "shop:waste", "shop:cafe", "shop:bakery", "tourism:viewpoint", "tourism:information", "tourism:attraction", "tourism:park", "tourism:picnic_site", "tourism:hotel", "tourism:hostel", "tourism:museum", "tourism:monument", "water:river", "water:stream", "water:lake", "water:pond", "water:canal", "water:bay", "water:waterfall", "water:reservoir", "water:marsh", "water:swamp", "water:salt_marsh", "water:floodplain", "water:beach", "waterway:river", "waterway:stream", "waterway:canal", "waterway:drift", "waterway:spring", "waterway:ditch", "waterway:sluice", "waterway:weir", "waterway:drain", "waterway:wastewater", "highway:crossing"
]

# 주요 태그(세부 분류 제외)를 추출하여 임베딩
def embed_tags(tags):
    # 주요 태그 추출: ':' 이후의 부분만 사용
    primary_tags = {tag: tag.split(":")[-1] for tag in tags}
    # 주요 태그를 임베딩
    tag_embeddings = {tag: model.encode(primary_tags[tag]) for tag in tags}
    return tag_embeddings

# 태그 임베딩 생성
tag_embeddings = embed_tags(osm_tags)

# 유사도 계산 및 가장 유사한 태그 찾기
def find_most_similar_tags(values, tag_embeddings, threshold=0.7):
    results = []
    for value in values:
        input_vec = model.encode(value)  # 입력 값 임베딩
        similarities = {tag: cosine_similarity([input_vec], [vec]).flatten()[0] for tag, vec in tag_embeddings.items()}
        sorted_tags = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

        # 임계값을 넘는 태그만 필터링
        matched_tags = [tag for tag, score in sorted_tags if score >= threshold]
        print(f"'{value}'와 유사한 태그들 (임계값 {threshold} 이상):", matched_tags)
        results.extend(matched_tags)  # 중첩 리스트가 아닌 단일 리스트에 추가
    return results

# JSON 업데이트 함수
# 설문 결과를 바탕으로 기본 가중치 설정
def update_features_with_similar_tags(features, tag_embeddings, threshold=0.7):
    if features['environment']:
        features['environment'] = find_most_similar_tags(features['environment'], tag_embeddings, threshold)
    else:
        features['environment'] = ['leisure:park']

    if features['dislike_environment']:
        features['dislike_environment'] = find_most_similar_tags(features['dislike_environment'], tag_embeddings, threshold)
    else:
        features['dislike_environment'] = ['highway:crossing']

    if features['facilities']:
        features['facilities'] = find_most_similar_tags(features['facilities'], tag_embeddings, threshold)
    else:
        features['facilities'] = ['amenity:toilets']

    if features['dislike_facilities']:
        features['dislike_facilities'] = find_most_similar_tags(features['dislike_facilities'], tag_embeddings, threshold)
    else:
        features['dislike_facilities'] = ['amenity:marketplace']

    # 환경과 시설에서 None만 남아있다면 기본 가중치 리스트로 초기화
    features['environment'] = [tag for tag in features['environment'] if tag] if features['environment'] else ['leisure:park']
    features['dislike_environment'] = [tag for tag in features['dislike_environment'] if tag] if features['dislike_environment'] else ['highway:crossing']
    features['facilities'] = [tag for tag in features['facilities'] if tag] if features['facilities'] else ['amenity:toilets']
    features['dislike_facilities'] = [tag for tag in features['dislike_facilities'] if tag] if features['dislike_facilities'] else ['amenity:marketplace']

    return features


# 업데이트된 특징 생성
user_preference = update_features_with_similar_tags(features, tag_embeddings)
print("user_preference:", json.dumps(user_preference, indent=2,ensure_ascii=False))


'forests'와 유사한 태그들 (임계값 0.7 이상): ['natural:forest']
user_preference: {
  "Distance": "2",
  "start": [
    {
      "name": "Ewha Womans University",
      "address": "이화여자대학교, 52, 이화여대길, 대현동, 신촌동, 서대문구, 서울특별시, 03760, 대한민국"
    }
  ],
  "destination": [
    {
      "name": "Ewha Womans University",
      "address": "이화여자대학교, 52, 이화여대길, 대현동, 신촌동, 서대문구, 서울특별시, 03760, 대한민국"
    }
  ],
  "incline": "low",
  "environment": [
    "natural:forest"
  ],
  "dislike_environment": [
    "highway:crossing"
  ],
  "facilities": [
    "amenity:toilets"
  ],
  "dislike_facilities": [
    "amenity:marketplace"
  ],
  "environment_distance_threshold": 300,
  "facility_distance_threshold": 300
}


In [24]:
import folium
import osmnx as ox
import networkx as nx
from osmnx import distance as ox_distance
from shapely.geometry import Point
from scipy.spatial import cKDTree
import random
import googlemaps

In [ ]:
# Google Maps API를 사용하여 출발지 좌표 가져오기
gmaps = googlemaps.Client(key='your-api-key')

start_geocode = gmaps.geocode(user_preference['start'], language='ko')
if not start_geocode:
    raise ValueError("Invalid start location.")
start_location = start_geocode[0]['geometry']['location']

if user_preference['destination'] != user_preference['start']:
  end_geocode = gmaps.geocode(user_preference['destination'], language='ko')
  if not start_geocode:
      raise ValueError("Invalid start location.")
  end_location = end_geocode[0]['geometry']['location']

In [26]:
# 태그 리스트를 딕셔너리로 변환하는 함수
def convert_tags_list_to_dict(tags_list):
    if isinstance(tags_list, list):
        tags_dict = {}
        for tag in tags_list:
            if ":" in tag:  # "key:value" 형식인지 확인
                key, value = tag.split(":")
                if key in tags_dict:
                    tags_dict[key].append(value)
                else:
                    tags_dict[key] = [value]
            else:
                # ":"가 없는 경우 원래 값을 유지
                if "original" not in tags_dict:
                    tags_dict["original"] = []
                tags_dict["original"].append(tag)
        return tags_dict
    else:
        # 리스트가 아닌 경우 원래 값을 반환
        return tags_list

# user_preference의 태그를 딕셔너리로 변환
def convert_user_preference_tags(user_preference):
    converted_tags = {}
    for key, tags_list in user_preference.items():
        # 리스트 형태의 태그만 변환, None 또는 문자열은 건너뜀
        if isinstance(tags_list, list):
            converted_tags[key] = convert_tags_list_to_dict(tags_list)
    return converted_tags

converted_tags = convert_user_preference_tags(user_preference)

#선호 값 가져오기
distance_tags = user_preference.get('Distance', {})
incline_tags = user_preference.get('incline', {})
environment_tags = converted_tags.get('environment', {})
dislike_environment_tags = converted_tags.get('dislike_environment', {})
facilities_tags = converted_tags.get('facilities', {})
dislike_facilities_tags = converted_tags.get('dislike_facilities', {})


In [27]:
# OSM에서 네트워크 그래프 불러오기 (회귀/단방향 경로 구분)
if user_preference['destination'] == user_preference['start']:
    # 회귀 경로의 경우: 기본 거리 (목표 거리 기준으로 반경 설정)
    if distance_tags is None or distance_tags in ['Not specified', '']:
        dist = 3000  # 기본 3km
    else:
        dist = float(distance_tags.replace('km', '').strip()) * 1000  # 입력 거리 변환
    G = ox.graph_from_point((start_location['lat'], start_location['lng']), dist=dist, network_type='walk')

    # 출발 노드 설정
    start_node = ox_distance.nearest_nodes(G, start_location['lng'], start_location['lat'])

else:
    # 단방향 경로의 경우: 출발지와 도착지 설정
    # 출발지 노드와 도착지 노드를 우선 기본 그래프에서 계산
    initial_dist = 3000  # 초기 그래프 생성 범위
    G_initial = ox.graph_from_point((start_location['lat'], start_location['lng']), dist=initial_dist, network_type='walk')

    # 출발 노드 설정
    start_node = ox_distance.nearest_nodes(G_initial, start_location['lng'], start_location['lat'])
    end_node = ox_distance.nearest_nodes(G_initial, end_location['lng'], end_location['lat'])

    # 최단 거리 계산
    try:
        shortest_distance = nx.shortest_path_length(G_initial, source=start_node, target=end_node, weight='length')
        print(f"출발지-도착지 최단 거리: {shortest_distance}m")
    except nx.NetworkXNoPath:
        raise ValueError("출발지와 도착지 사이에 경로가 없습니다.")

    # 최단 거리 + 2km 반경으로 그래프 생성
    extended_dist = shortest_distance + 2000
    G = ox.graph_from_point((start_location['lat'], start_location['lng']), dist=extended_dist, network_type='walk')

    # 새 그래프에서 출발 노드와 도착 노드 다시 설정
    start_node = ox_distance.nearest_nodes(G, start_location['lng'], start_location['lat'])
    end_node = ox_distance.nearest_nodes(G, end_location['lng'], end_location['lat'])

# 공통 작업 이후의 G와 start_node는 올바르게 설정되어 있음


/home/yim/.local/lib/python3.10/site-packages/osmnx/graph.py:191: FutureWarning: The expected order of coordinates in `bbox` will change in the v2.0.0 release to `(left, bottom, right, top)`.
  G = graph_from_bbox(


In [28]:
# 고도 데이터 추가 함수
def add_elevation_data_batch(G):
    nodes = list(G.nodes)
    node_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in nodes]
    elevation_results = []
    for i in range(0, len(node_coords), 512):
        batch = node_coords[i:i + 512]
        elevation_results.extend(gmaps.elevation(batch))
    for node, elevation_result in zip(nodes, elevation_results):
        G.nodes[node]['elevation'] = elevation_result['elevation']

# 경사도 가중치 계산 함수
def apply_incline_weights(G, user_preference):
    for u, v, data in G.edges(data=True):
        elevation_diff = abs(G.nodes[v].get('elevation', 0) - G.nodes[u].get('elevation', 0))
        incline = elevation_diff / data.get('length', 1)
        if user_preference['incline'] == 'low' and incline > 0.02:
            data['incline_weight'] = 1.5
        elif user_preference['incline'] == 'high' and incline < 0.02:
            data['incline_weight'] = 1.5
        else:
            data['incline_weight'] = 1.0
    return G

# 시설 가중치 계산 함수
def compute_nearest_facilities_kdtree(G, facilities_data, disliked_data, threshold):
    facility_points = [
        (geom.centroid.y, geom.centroid.x) if geom.is_valid and hasattr(geom, "centroid") else None
        for geom in facilities_data.geometry
    ]
    facility_points = [point for point in facility_points if point is not None]
    facility_tree = cKDTree(facility_points) if facility_points else None

    disliked_points = [
        (geom.centroid.y, geom.centroid.x) if geom.is_valid and hasattr(geom, "centroid") else None
        for geom in disliked_data.geometry
    ]
    disliked_points = [point for point in disliked_points if point is not None]
    disliked_tree = cKDTree(disliked_points) if disliked_points else None

    for node, data in G.nodes(data=True):
        node_point = (data['y'], data['x'])
        _,facility_distance = facility_tree.query(node_point)
        _,disliked_distance = disliked_tree.query(node_point)

        if facility_distance <= threshold:
            data['facility_weight'] = 0.7
        elif disliked_distance <= threshold:
            data['facility_weight'] = 2.0  # 비선호 시설에 높은 가중치
        else:
            data['facility_weight'] = 1.0
    return G


# 환경 가중치 계산 함수
def apply_environment_weights(G, user_preference, environment_tags, threshold):
    environment_mapping = environment_tags
    preferred_tags = set()
    disliked_tags = set()

    # 선호 환경 태그 추가 (None 방지)
    environment_preference = user_preference.get("environment", [])
    if environment_preference:
        for env in environment_preference:
            if env in environment_mapping:
                preferred_tags.update(environment_mapping[env])

    # 비선호 환경 태그 추가 (None 방지)
    dislike_environment_preference = user_preference.get("dislike_environment", [])
    if dislike_environment_preference:
        for env in dislike_environment_preference:
            if env in environment_mapping:
                disliked_tags.update(environment_mapping[env])

    preferred_data = []
    for tag in preferred_tags:
        key, value = tag.split(":")
        env_nodes = ox.features_from_place("Seoul, South Korea", tags={key: value})
        if not env_nodes.empty:
            preferred_data.extend([
                (geom.centroid.y, geom.centroid.x) for geom in env_nodes.geometry
                if geom.is_valid and hasattr(geom, "centroid")
            ])
    if not preferred_data:
        return G
    preferred_tree = cKDTree(preferred_data)

    disliked_data = []
    for tag in disliked_tags:
        key, value = tag.split(":")
        env_nodes = ox.features_from_place("Seoul, South Korea", tags={key: value})
        if not env_nodes.empty:
            disliked_data.extend([
                (geom.centroid.y, geom.centroid.x) for geom in env_nodes.geometry
                if geom.is_valid and hasattr(geom, "centroid")
            ])
    if not disliked_data:
        return G
    disliked_tree = cKDTree(disliked_data)

    # 노드별 환경 가중치 계산
    for node, data in G.nodes(data=True):
        node_point = (data['y'], data['x'])
        _, preferred_distance = preferred_tree.query(node_point)
        _, disliked_distance = disliked_tree.query(node_point)

        if preferred_distance <= threshold:
            data['environment_weight'] = 0.7
        elif disliked_distance <= threshold:
            data['environment_weight'] = 2.0  # 비선호 환경에 높은 가중치
        else:
            data['environment_weight'] = 1.0

    return G


# 최종 가중치 계산 함수
def calculate_final_weights(G):
    for u, v, data in G.edges(data=True):
        # 간선의 양 끝 노드에서 facility_weight와 environment_weight 가져오기
        facility_weight_u = G.nodes[u].get('facility_weight', 1.0)
        facility_weight_v = G.nodes[v].get('facility_weight', 1.0)
        environment_weight_u = G.nodes[u].get('environment_weight', 1.0)
        environment_weight_v = G.nodes[v].get('environment_weight', 1.0)
        
        # 양 끝 노드의 가중치 평균 계산
        facility_weight_avg = (facility_weight_u + facility_weight_v) / 2
        environment_weight_avg = (environment_weight_u + environment_weight_v) / 2
        
        # 최종 가중치 계산
        data['weight'] = data.get('incline_weight', 1.0) * facility_weight_avg * environment_weight_avg

        # 디버깅 출력
        print(f"Edge: ({u}, {v})")
        print(f"  Incline Weight: {data.get('incline_weight', 1.0)}")
        print(f"  Facility Weight Avg: {facility_weight_avg}")
        print(f"  Environment Weight Avg: {environment_weight_avg}")
        print(f"  Final Weight: {data['weight']}")
    return G

In [29]:
#회귀 경로 탐색 함수
from tqdm import tqdm

# 회귀 아닐 경우 경로 탐색 함수
def find_routes_of_distance(G, start_node, target_distance=5000, tolerance=500):
    print("회귀 아닌 경우")
    lengths, paths = nx.single_source_dijkstra(G, source=start_node, weight='length')
    target_nodes = [
        (node, length, paths[node]) for node, length in lengths.items()
        if target_distance - tolerance <= length <= target_distance + tolerance
    ]
    if not target_nodes:
        raise ValueError("5km에 해당하는 노드를 찾을 수 없습니다.")
    return random.sample(target_nodes, min(10, len(target_nodes)))


# 회귀 아닐 경우 가중치 계산
def calculate_route_weight(G, path):
    # 경로에 포함된 노드의 가중치 계산
    weights = [
        G.nodes[node].get('incline_weight', 1.0) *
        G.nodes[node].get('facility_weight', 1.0) *
        G.nodes[node].get('environment_weight', 1.0)
        for node in path
    ]

    # 평균 가중치 반환
    return sum(weights) / len(weights)




# 회귀 경로 탐색 함수
def find_round_trip_routes(G, start_node, target_distance=5000, tolerance=500):
    print("회귀")
    """
    출발지 → 도착지 → 출발지 회귀 경로를 찾는 함수.
    동일한 경로를 피하면서 왕복 경로를 구성.
    """
    lengths, paths = nx.single_source_dijkstra(G, source=start_node, weight='length')
    target_nodes = [
        (node, length, paths[node]) for node, length in lengths.items()
        if target_distance - tolerance <= length <= target_distance + tolerance
    ]
    if not target_nodes:
        raise ValueError("5km에 해당하는 노드를 찾을 수 없습니다.")

    selected_targets = random.sample(target_nodes, min(10, len(target_nodes)))
    round_trip_routes = []
    for target_node, forward_distance, forward_path in selected_targets:
        G_temp = G.copy()
        for u, v in zip(forward_path[:-1], forward_path[1:]):
            if G_temp.has_edge(u, v):
                G_temp.remove_edge(u, v)
            if G_temp.has_edge(v, u):
                G_temp.remove_edge(v, u)

        try:
            print("start node",start_node)
            backward_path = nx.shortest_path(G_temp, source=target_node, target=start_node, weight='length')
            backward_distance = sum(
                G[u][v][0]['length'] for u, v in zip(backward_path[:-1], backward_path[1:])
            )
            print("try")
            round_trip_routes.append((forward_path, forward_distance, backward_path, backward_distance))
        except nx.NetworkXNoPath:
            print("실패")
            continue

    return round_trip_routes


def calculate_round_trip_weight(G, forward_path, backward_path):
    # Forward path weights
    forward_weights = []
    print("\nForward Path Weights:")
    for u, v in zip(forward_path[:-1], forward_path[1:]):
        incline_weight = G[u][v][0].get('incline_weight', 1.0)
        facility_weight = G.nodes[u].get('facility_weight', 1.0)
        environment_weight = G.nodes[u].get('environment_weight', 1.0)
        weight = incline_weight * facility_weight * environment_weight
        forward_weights.append(weight)
        print(f"  Edge ({u} -> {v}):")
        print(f"    Incline Weight: {incline_weight}")
        print(f"    Facility Weight: {facility_weight}")
        print(f"    Environment Weight: {environment_weight}")
        print(f"    Combined Weight: {weight}")

    # Backward path weights
    backward_weights = []
    print("\nBackward Path Weights:")
    for u, v in zip(backward_path[:-1], backward_path[1:]):
        incline_weight = G[u][v][0].get('incline_weight', 1.0)
        facility_weight = G.nodes[u].get('facility_weight', 1.0)
        environment_weight = G.nodes[u].get('environment_weight', 1.0)
        weight = incline_weight * facility_weight * environment_weight
        backward_weights.append(weight)
        print(f"  Edge ({u} -> {v}):")
        print(f"    Incline Weight: {incline_weight}")
        print(f"    Facility Weight: {facility_weight}")
        print(f"    Environment Weight: {environment_weight}")
        print(f"    Combined Weight: {weight}")

    # 마지막 노드의 facility_weight와 environment_weight 추가
    final_node_forward = forward_path[-1]
    final_node_backward = backward_path[-1]
    forward_weights.append(
        G.nodes[final_node_forward].get('facility_weight', 1.0) *
        G.nodes[final_node_forward].get('environment_weight', 1.0)
    )
    backward_weights.append(
        G.nodes[final_node_backward].get('facility_weight', 1.0) *
        G.nodes[final_node_backward].get('environment_weight', 1.0)
    )
    print(f"\nFinal Node (Forward): {final_node_forward}")
    print(f"  Facility Weight: {G.nodes[final_node_forward].get('facility_weight', 1.0)}")
    print(f"  Environment Weight: {G.nodes[final_node_forward].get('environment_weight', 1.0)}")
    print(f"\nFinal Node (Backward): {final_node_backward}")
    print(f"  Facility Weight: {G.nodes[final_node_backward].get('facility_weight', 1.0)}")
    print(f"  Environment Weight: {G.nodes[final_node_backward].get('environment_weight', 1.0)}")

    # 평균 가중치 계산
    total_weights = forward_weights + backward_weights
    average_weight = sum(total_weights) / len(total_weights)

    print(f"\nAverage Weight for Round Trip Path: {average_weight}")
    return average_weight




In [30]:
# 고도 데이터 추가
add_elevation_data_batch(G)

import pandas as pd  # pandas를 임포트
import geopandas as gpd
import osmnx as ox

# 경로 초기화
forward_path = []
backward_path =[]
optimal_route = []


# 시설 및 비선호 시설 데이터 가져오기
facilities_data = ox.features_from_place("Seoul, South Korea", facilities_tags)
disliked_facilities_data = ox.features_from_place("Seoul, South Korea",dislike_facilities_tags)


# 가중치 계산 적용
G = apply_incline_weights(G, user_preference)
G = compute_nearest_facilities_kdtree(G, facilities_data, disliked_facilities_data, threshold=300)
G = apply_environment_weights(G, user_preference,environment_tags, threshold=300)
G = calculate_final_weights(G)


if user_preference['destination'] == user_preference['start']:
  # 왕복 경로 탐색
  try:
      round_trip_routes = find_round_trip_routes(G, start_node, target_distance=dist, tolerance=500)
      print("왕복")
  except ValueError as e:
      print(e)
      exit()

  # 왕복 경로별 최종 가중치 계산
  round_trip_weights = []
  print("confirm",enumerate(round_trip_routes))
  for idx, (forward_path, forward_distance, backward_path, backward_distance) in enumerate(round_trip_routes):
      weight = calculate_round_trip_weight(G, forward_path, backward_path)
      print("최종 weight:",weight)
      total_distance = forward_distance + backward_distance
      round_trip_weights.append((idx, weight, total_distance, forward_path, backward_path))

  # 최종 가중치가 낮은 상위 3개의 경로 선택
  top_3_round_trips = sorted(round_trip_weights, key=lambda x: x[1])[:3]

  # 색상 목록 (각 왕복 경로별 색상 설정)
  colors = ['red', 'green', 'blue']

  # Folium 지도 생성
  m = folium.Map(location=[start_location['lat'], start_location['lng']], zoom_start=14)
  folium.Marker([start_location['lat'], start_location['lng']], tooltip="출발지", icon=folium.Icon(color="black")).add_to(m)

  # 상위 3개 왕복 경로를 지도에 추가
  for idx, (route_idx, weight, total_distance, forward_path, backward_path) in enumerate(top_3_round_trips):
      # 출발지 → 도착지 경로
      forward_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in forward_path]
      folium.PolyLine(
          forward_coords,
          color=colors[idx],
          #weight=5,
          opacity=0.7,
          tooltip=f"Forward Route {idx + 1} (Weight: {weight:.2f}, Distance: {total_distance:.2f}m)"
      ).add_to(m)

      # 도착지 → 출발지 경로
      backward_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in backward_path]
      folium.PolyLine(
          backward_coords,
          color=colors[idx],
          #weight=5,
          opacity=0.7,
          dash_array="5, 5",  # 점선 표시
          tooltip=f"Backward Route {idx + 1} (Weight: {weight:.2f}, Distance: {total_distance:.2f}m)"
      ).add_to(m)

      # 도착지 마커 추가
      target_node = forward_path[-1]
      target_lat, target_lon = G.nodes[target_node]['y'], G.nodes[target_node]['x']
      folium.Marker(
          [target_lat, target_lon],
          tooltip=f"Destination {idx + 1}",
          icon=folium.Icon(color=colors[idx])
      ).add_to(m)

  # 지도 저장
  m.save("top_3_round_trip_routes.html")
  print("지도 파일이 top_3_round_trip_routes.html로 저장되었습니다.")

  # 최종 결과 출력
  print("\nTop 3 Round Trip Routes with Lowest Weights:")
  for idx, (route_idx, weight, total_distance, forward_path, backward_path) in enumerate(top_3_round_trips):
      print(f"\nRoute {idx + 1}:")
      print(f"  Original Index: {route_idx}")
      print(f"  Final Weight: {weight:.2f}")
      print(f"  Total Distance: {total_distance:.2f}m")
      print(f"  Forward Path: {forward_path}")
      print(f"  Backward Path: {backward_path}")



else:
  # 출발지와 목적지 노드 찾기
  start_node = ox_distance.nearest_nodes(G, start_location['lng'], start_location['lat'])
  end_node = ox_distance.nearest_nodes(G, end_location['lng'], end_location['lat'])

  # 최적 경로 계산
  optimal_route = nx.shortest_path(G, start_node, end_node, weight='weight')

  # Folium 지도 생성
  m = folium.Map(location=[start_location['lat'], start_location['lng']], zoom_start=14)
  route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in optimal_route]
  folium.PolyLine(route_coords, color="blue", weight=5, opacity=0.7).add_to(m)
  folium.Marker([start_location['lat'], start_location['lng']], tooltip="출발지").add_to(m)
  folium.Marker([end_location['lat'], end_location['lng']], tooltip="도착지").add_to(m)
  m.save("optimal_route_map.html")
  print("지도 파일이 optimal_route_map.html로 저장되었습니다.")

  # 경로에 포함된 노드 출력
  print("\nNodes in the optimal route:")
  for node in optimal_route:
      print(f"Node ID: {node}, Coordinates: (Lat: {G.nodes[node]['y']}, Lon: {G.nodes[node]['x']})")

  # 경로에 포함된 간선 출력
  print("\nEdges in the optimal route:")
  for i in range(len(optimal_route) - 1):
      u, v = optimal_route[i], optimal_route[i + 1]
      edge_data = G[u][v][0]  # 간선의 첫 번째 데이터
      print(f"Edge: ({u}, {v})")
      print(f"  Length: {edge_data.get('length', 'Unknown')}")
      print(f"  Incline Weight: {edge_data.get('incline_weight', 'Unknown')}")
      print(f"  Final Weight: {edge_data.get('weight', 'Unknown')}")

  # 탐색한 전체 노드 수와 경로에 포함된 노드 수 출력
  print("\nSummary:")
  print(f"Total number of nodes in the graph: {len(G.nodes)}")
  print(f"Number of nodes in the optimal route: {len(optimal_route)}")

Edge: (355173024, 11296428354)
  Incline Weight: 1.0
  Facility Weight Avg: 2.0
  Environment Weight Avg: 1.0
  Final Weight: 2.0
Edge: (355173024, 11257338989)
  Incline Weight: 1.0
  Facility Weight Avg: 2.0
  Environment Weight Avg: 1.0
  Final Weight: 2.0
Edge: (355173024, 11257338599)
  Incline Weight: 1.0
  Facility Weight Avg: 2.0
  Environment Weight Avg: 1.0
  Final Weight: 2.0
Edge: (357932352, 4427725735)
  Incline Weight: 1.5
  Facility Weight Avg: 0.7
  Environment Weight Avg: 1.0
  Final Weight: 1.0499999999999998
Edge: (414683302, 3893042401)
  Incline Weight: 1.5
  Facility Weight Avg: 0.7
  Environment Weight Avg: 1.0
  Final Weight: 1.0499999999999998
Edge: (414683302, 8154128184)
  Incline Weight: 1.5
  Facility Weight Avg: 0.7
  Environment Weight Avg: 1.0
  Final Weight: 1.0499999999999998
Edge: (414683302, 7068970634)
  Incline Weight: 1.5
  Facility Weight Avg: 0.7
  Environment Weight Avg: 1.0
  Final Weight: 1.0499999999999998
Edge: (414685214, 5987800033)
  In

In [31]:
print(forward_path)
print(backward_path)

print(optimal_route)

[3785057860, 5231979558, 5728608373, 4973475979, 3785056723, 436859669, 2329648805, 3864845856, 2329648800, 3864845843, 436859682, 5728608390, 2329648803, 2329648796, 3864845840, 4591019744, 9283138047, 9283138058, 9283138057, 9283138056, 9283138055, 9283138070, 436718687, 11080341314, 9283138075, 9283138074, 9283138073, 9283138095, 9283138094, 4121393430, 4590191568, 9282393208, 2775071293, 9282393210, 436859638, 6048920035, 6048920040, 6048920031, 6048920029, 6048920028, 6048920023, 4683091435]
[4683091435, 4683091436, 8477575613, 6048920026, 6048920044, 6048920034, 6048920038, 4591702787, 436874458, 9282393211, 8537159258, 8537159257, 8537159256, 9283138088, 9283138101, 9283138089, 5887231074, 9944378196, 9944378212, 11080341314, 9468334143, 9283051968, 3617050506, 5235961965, 2364556168, 12107957060, 12107957059, 3309835000, 3864845891, 3310757830, 3618648887, 3309835035, 3618648996, 3309835041, 3309835047, 3864845151, 9703439616, 3864845156, 9703439609, 436841150, 3864845761, 4368

In [32]:
import json

# 데이터 초기화
forward_paths_coords = []  # 모든 Forward 경로의 위도-경도 리스트
backward_paths_coords = []  # 모든 Backward 경로의 위도-경도 리스트
optimal_route_coords = []  # 최적 경로의 위도-경도 리스트

# 경로 처리
if 'optimal_route' in globals() and optimal_route:
    # Optimal Route가 존재하는 경우
    optimal_route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in optimal_route]
    optimal_route_coords = [list(coord) for coord in optimal_route_coords]  # 튜플을 리스트로 변환
    print("\nOptimal Route:")
    print(f"  Optimal Route (Lat, Lng): {optimal_route_coords}")

elif 'top_3_round_trips' in globals() and top_3_round_trips:
    # Forward와 Backward 경로가 존재하는 경우
    print("\nTop 3 Round Trip Routes with Lowest Weights:")
    for idx, (route_idx, weights, total_distance, forward_path, backward_path) in enumerate(top_3_round_trips):
        # Forward Path를 위도-경도 형식으로 변환
        forward_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in forward_path]
        forward_coords = [list(coord) for coord in forward_coords]  # 튜플을 리스트로 변환
        forward_paths_coords.append(forward_coords)  # 변환된 Forward 경로 저장

        # Backward Path를 위도-경도 형식으로 변환
        backward_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in backward_path]
        backward_coords = [list(coord) for coord in backward_coords]  # 튜플을 리스트로 변환
        backward_paths_coords.append(backward_coords)  # 변환된 Backward 경로 저장

        print(f"Route {idx + 1}:")
        print(f"  Forward Path (Lat, Lng): {forward_coords}")
        print(f"  Backward Path (Lat, Lng): {backward_coords}")
else:
    print("No valid paths or routes found.")


# 애니메이션 추가
html_filename = "animated_routes.html"
with open(html_filename, "w", encoding="utf-8") as f:
    f.write(f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css" />
    <style>
        /* Forward 마커 스타일 */
        .forward-marker img {{
            background: none !important; /* 배경 제거 */
            filter: opacity(1);         /* 투명도 유지 */
            -webkit-filter: opacity(1); /* Safari 지원 */
        }}

        /* Backward 마커 스타일 */
        .backward-marker img {{
            background: none !important;
            filter: opacity(1);
            -webkit-filter: opacity(1);
        }}
    </style>
</head>
<body>
    <div id="map" style="height: 100vh;"></div>
    <script>
        var map = L.map('map').setView([37.5618588, 126.9468339], 14);
        L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{
            maxZoom: 18,
            attribution: '&copy; OpenStreetMap contributors'
        }}).addTo(map);

        // Forward and Backward Paths for round trip
        var forwardPaths = {json.dumps(forward_paths_coords if 'forward_paths_coords' in globals() else [])};
        var backwardPaths = {json.dumps(backward_paths_coords if 'backward_paths_coords' in globals() else [])};

        // Optimal Path for one-way
        var optimalPath = {json.dumps([(G.nodes[node]['y'], G.nodes[node]['x']) for node in optimal_route] if 'optimal_route' in globals() else [])};

        var colors = ['orange', 'green', 'blue'];

        // 커스텀 아이콘 정의
        var forwardIcon = L.icon({{
            iconUrl: 'https://www.pngarts.com/files/11/Cartoon-Hello-Kitty-PNG-Picture.png', // Forward 마커 PNG 이미지 경로
            iconSize: [40, 40],                       // 아이콘 크기
            iconAnchor: [16, 32],                     // 기준점
            className: 'forward-marker'               // 사용자 정의 클래스
        }});

        var backwardIcon = L.icon({{
            iconUrl: 'https://www.pngarts.com/files/11/Cartoon-Hello-Kitty-PNG-Picture.png', // Backward 마커 PNG 이미지 경로
            iconSize: [40, 40],                        // 아이콘 크기
            iconAnchor: [16, 16],                      // 기준점
            className: 'backward-marker'               // 사용자 정의 클래스
        }});

        function animatePath(path, color, icon, dashArray = null) {{
            return new Promise((resolve) => {{
                if (path && path.length > 0) {{
                    var marker = L.marker(path[0], {{ icon: icon }}).addTo(map);
                    var i = 0;

                    function moveMarker() {{
                        if (i < path.length) {{
                            marker.setLatLng(path[i]); // 마커 위치 업데이트
                            i++;
                            setTimeout(moveMarker, 200); // 100ms 간격으로 이동
                        }} else {{
                            marker.remove(); // 애니메이션 종료 후 마커 제거
                            resolve(); // 애니메이션 완료 시 resolve 호출
                        }}
                    }}
                    moveMarker();

                    // 경로를 Polyline으로 시각화
                    L.polyline(path, {{ color: color, weight: 8, dashArray: dashArray }}).addTo(map);
                }} else {{
                    console.warn("Invalid path:", path);
                    resolve(); // 비어있는 경로는 즉시 완료 처리
                }}
            }});
        }}

        async function animateForwardAndBackward(forwardPath, backwardPath, color) {{
            // Forward Path Animation
            await animatePath(forwardPath, color, forwardIcon);

            // Backward Path Animation
            await animatePath(backwardPath, color, backwardIcon, '10, 15'); // 점선 길이와 간격 설정
        }}

        async function startAnimations() {{
            // 왕복 경로 애니메이션
            forwardPaths.forEach((forwardPath, index) => {{
                var backwardPath = backwardPaths[index] || [];
                animateForwardAndBackward(forwardPath, backwardPath, colors[index % colors.length]);
            }});

            // 최적 경로 애니메이션 (출발지 → 도착지)
            if (optimalPath.length > 0) {{
                await animatePath(optimalPath, 'blue', forwardIcon);
            }}
        }}

        // 전체 경로를 먼저 시각화
        forwardPaths.forEach((path, index) => {{
            L.polyline(path, {{ color: colors[index % colors.length], weight: 8 }}).addTo(map);
        }});
        backwardPaths.forEach((path, index) => {{
            L.polyline(path, {{ color: colors[index % colors.length], weight: 8, dashArray: '10, 15' }}).addTo(map);
        }});
        if (optimalPath.length > 0) {{
            L.polyline(optimalPath, {{ color: 'blue', weight: 8 }}).addTo(map);
        }}

        // 애니메이션 시작
        startAnimations();
    </script>
</body>
</html>
""")



Top 3 Round Trip Routes with Lowest Weights:
Route 1:
  Forward Path (Lat, Lng): [[37.5621944, 126.9468733], [37.5620042, 126.9472315], [37.5601943, 126.9456287], [37.5599689, 126.9456175], [37.5598574, 126.9455379], [37.559188, 126.9456887], [37.5592145, 126.9464958], [37.5592218, 126.946733], [37.5592248, 126.9468314], [37.5592354, 126.9471767], [37.5592413, 126.9473682], [37.5592794, 126.9474884], [37.5593607, 126.9477964], [37.5594193, 126.9479665], [37.5594271, 126.9480896], [37.5594378, 126.9482563], [37.5594535, 126.9485024], [37.5595306, 126.9487145], [37.5595334, 126.9488075], [37.5596064, 126.9490662], [37.5596007, 126.9491402], [37.5597757, 126.9498684], [37.5599438, 126.9501709], [37.5600953, 126.9504434], [37.5602816, 126.9507395], [37.5606393, 126.9512686], [37.5610592, 126.9518894], [37.5615676, 126.9528475], [37.5620487, 126.9551375], [37.5621874, 126.9556762], [37.5622556, 126.9556482], [37.5625125, 126.9564244], [37.5628466, 126.9574413], [37.5631359, 126.9573055], [

In [33]:
m

###pip install

In [34]:
# pip install osmnx==1.9.4
# numpy-1.26.4 

In [35]:
# pip install googlemaps

In [36]:
# pip install openai==0.28

In [37]:
# pip install spacy

In [38]:
# pip install googlemaps